# Dataset-to-Report LLM Pipeline

This module loads a dataset and delegates the computation of statistical or causal effects to a large language model. The model analyzes the data, performs the necessary mathematical reasoning, and generates a structured report summarizing the estimated effects and their interpretation.

In [1]:
import json
import importlib

import os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI, BadRequestError
import reportlab
import sys
from pathlib import Path
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import src.llm as llm
from src.llm import generate_report_from_file_id

In [2]:
dataset2_config = {
    "x": "Gender",
    "y": "Admission",
    "z": ["Major"],
    "w": [],
    "x0": "F",
    "x1": "M",
    "y_value": "Accepted",
    "w_values": None,
}

In [3]:
data_path = REPO_ROOT / "data" / "processed" / "berkeley_filtered.txt"
uploaded = client.files.create(
    file=open(data_path, "rb"),
    purpose="assistants",
)

text, latex = generate_report_from_file_id(uploaded.id, client,prompt_kwargs=dataset2_config)



USER_PROMPT: 
You are given a dataset as a PDF rendering of the dataset (table).

Column roles:
- X (protected variable): <Gender>, with x0=F and x1=M
- Y (outcome attribute): <Admission> y='Accepted'
- Z (spurious features): <Major>
- W (mediator variables): <None>

Your task:
1. Analyze the dataset.
2. Compute the fairness decomposition according to Bareinboim and Plecko theory, both general and X-Z specific effects.
3. Produce a structured report.

Output format MUST be:

TEXT:
<plain language report>

LATEX:
<standalone LaTeX document>

USAGE: ResponseUsage(input_tokens=9736, input_tokens_details=InputTokensDetails(cached_tokens=1792), output_tokens=8362, output_tokens_details=OutputTokensDetails(reasoning_tokens=5952), total_tokens=18098)


In [4]:
print(latex)

\documentclass{article}
\usepackage{geometry}
\geometry{margin=1in}
\usepackage{amsmath}
\begin{document}
Title: "Fairness Decomposition Report"

\subsection*{Overview of the Fairness Analysis}
This analysis decomposes the difference in admission rates between the protected groups Gender $X$: $x0=\text{F}$ and $x1=\text{M}$ for the outcome Admission (Accepted). We treat Major as the spurious/confounding variable $Z$ and there are no mediators $W$. The decomposition separates the observed group disparity into total variation, the total causal effect, direct and indirect components (no indirect effect here), and the spurious component attributable to differences in Major.

\subsection*{Decomposition of Effects}
\begin{itemize}
\item Total variation (observational difference): $P(\text{Accepted}\mid \text{Sex}=\text{M}) - P(\text{Accepted}\mid \text{Sex}=\text{F}) = 0.2322$. Interpretation: Males have a higher observed acceptance rate than Females by about 23.22 percentage points.
\item T

## Another example of llm only 

Using student_mat.csv dataset.

In [5]:
data_path = REPO_ROOT / "data" / "processed" / "student_mat.txt"



df = pd.read_csv(data_path)

selected_columns = [
    "S2_sex",
    "T_grade",
    "studytime",
]

df[selected_columns].head()



,S2_sex,T_grade,studytime
0,F,0,2
1,F,0,2
2,F,1,2
3,F,1,3
4,F,1,2


In [6]:
dataset1_config = {
    "x": "Sex",
    "y": "T_grade",
    "z": None,
    "w": ["studytime"],
    "x0": "F",
    "x1": "M",
    "y_value": "1",
    "w_values": ["1", "2", "3", "4"],
}

In [7]:

uploaded = client.files.create(
    file=open(data_path, "rb"),
    purpose="assistants",
)

text, latex = generate_report_from_file_id(uploaded.id, client,prompt_kwargs=dataset1_config)



USER_PROMPT: 
You are given a dataset as a PDF rendering of the dataset (table).

Column roles:
- X (protected variable): <Sex>, with x0=F and x1=M
- Y (outcome attribute): <T_grade> y='1'
- Z (spurious features): <None>
- W (mediator variables): <studytime> ['1', '2', '3', '4']

Your task:
1. Analyze the dataset.
2. Compute the fairness decomposition according to Bareinboim and Plecko theory, both general and X-Z specific effects.
3. Produce a structured report.

Output format MUST be:

TEXT:
<plain language report>

LATEX:
<standalone LaTeX document>

USAGE: ResponseUsage(input_tokens=6713, input_tokens_details=InputTokensDetails(cached_tokens=1792), output_tokens=6361, output_tokens_details=OutputTokensDetails(reasoning_tokens=3712), total_tokens=13074)


In [8]:
print(latex)

\documentclass{article}
\usepackage{geometry}
\geometry{margin=1in}
\usepackage{amsmath}
\begin{document}
Title: "Fairness Decomposition Report"

\subsection*{Overview of the Fairness Analysis}
This analysis decomposes the difference in the outcome $T\_grade$ (binary indicator equal to 1) between groups defined by Sex: $X=x0$ (F) versus $X=x1$ (M). The single mediator is \texttt{studytime} with ordered levels $\{1,2,3,4\}$ and no spurious confounders $Z$ were provided. We decompose the observed group disparity into total variation, total causal effect, indirect effect via \texttt{studytime}, direct effect, and spurious effect. All components are estimated from the sample frequencies and conditional means in the supplied data.

\subsection*{Decomposition of Effects}
\begin{itemize}
\item Total variation:
  \begin{itemize}
  \item $P(T\_grade=1\mid X=\text{M}) - P(T\_grade=1\mid X=\text{F}) = 0.7059 - 0.6394 = 0.0665$.
  \item Interpretation: Males have about a $0.0665$ higher probabilit